# 02 — MRI Preprocessing

Loads Baby Open Brains BIDS dataset (10 T1w/T2w subjects), selects T2w for infants < 12 months, extracts axial/coronal/sagittal slices at center of mass, resizes to 64×64, saves `(3, 64, 64)` arrays to `datasets/processed/mri/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Data Loader (Supports DVC on Mounted Google Drive, Drag-and-Drop, or Auto Download)
import os, glob, shutil
os.makedirs('datasets/mri/baby_open_brains', exist_ok=True)

# OPTION A: If your DVC cache is in your own Google Drive, point DVC to your mounted drive!
# => Update this to the exact folder path of your DVC storage inside your Google Drive
MY_DRIVE_DVC_PATH = '/content/drive/MyDrive/DVC/Earlyminds/earlyMind_DVC' # <-- Updated to your exact folder!

if os.path.exists(MY_DRIVE_DVC_PATH):
    print(f'Found local DVC remote at {MY_DRIVE_DVC_PATH}...')
    os.system(f'dvc remote add -d local_gdrive {MY_DRIVE_DVC_PATH} --force')
    os.system('dvc pull')
    print('DVC pull complete from mounted Google Drive!')

# OPTION B: Check if user dragged-and-dropped the raw files directly to /content/
if len(glob.glob('/content/*.nii.gz')) > 0:
    print('Found raw files in /content/! Moving them to datasets/mri/baby_open_brains...')
    os.system(f'mv /content/*.nii.gz datasets/mri/baby_open_brains/')
    os.system(f'mv /content/*.tsv datasets/mri/baby_open_brains/ 2>/dev/null')

# 2. Check if user dragged-and-dropped the .tar.gz file to /content/
elif os.path.exists('/content/mri_raw.tar.gz'):
    print('Found mri_raw.tar.gz in /content/! Extracting...')
    os.system(f'tar -xzf /content/mri_raw.tar.gz -C datasets/mri')

# 3. Fallback: Automatically download from GitHub Releases
else:
    print('Downloading dataset automatically from GitHub Releases...')
    os.system(f'wget -qO mri_raw.tar.gz https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data/mri_raw.tar.gz')
    os.system(f'tar -xzf mri_raw.tar.gz -C datasets/mri')

print('Datasets ready.')


In [ ]:
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
from src.data.mri_loader import process_mri_dataset
import pandas as pd
import numpy as np

print('MRI raw dir: ', cfg.paths.mri_raw)
print('MRI proc dir:', cfg.paths.mri_processed)

# Inspect participants.tsv
tsv_path = cfg.paths.mri_raw / 'participants.tsv'
if tsv_path.exists():
    df_part = pd.read_csv(tsv_path, sep='\t')
    print('Participants TSV columns:', df_part.columns.tolist())
    display(df_part)
else:
    print('participants.tsv not found')

In [ ]:
# Run full MRI preprocessing
results = process_mri_dataset(
    mri_dir=cfg.paths.mri_raw,
    output_dir=cfg.paths.mri_processed,
)

print('\n=== Results ===')
for pid, info in results.items():
    print(f'  {pid}: slices={info["slices"].shape}, age={info["age_months"]:.1f}mo, '
          f'label={info["label"]}, DQ={info["dq"]:.1f}')

In [ ]:
# Visualize slices for first subject
import matplotlib.pyplot as plt
import numpy as np

first_pid = list(results.keys())[0]
slices = results[first_pid]['slices']  # (3, 64, 64)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ['Axial', 'Coronal', 'Sagittal']
for i, (ax, title) in enumerate(zip(axes, titles)):
    ax.imshow(slices[i], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{first_pid} — {title}', fontsize=12)
    ax.axis('off')
plt.tight_layout()
cfg.paths.reports.mkdir(parents=True, exist_ok=True)
plt.savefig(cfg.paths.reports / 'mri_slices.png', dpi=100)
plt.show()
print(f'Slice shape: {slices.shape}, range: [{slices.min():.3f}, {slices.max():.3f}]')

In [ ]:
# Show augmented delayed myelination example
from src.data.mri_loader import simulate_delayed_myelination
import numpy as np

rng = np.random.default_rng(42)
delayed = simulate_delayed_myelination(slices, rng=rng)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    axes[0, i].imshow(slices[i],  cmap='gray')
    axes[0, i].set_title(f'Normal — {titles[i]}')
    axes[1, i].imshow(delayed[i], cmap='gray')
    axes[1, i].set_title(f'Simulated Delay — {titles[i]}')
    for ax in [axes[0, i], axes[1, i]]:
        ax.axis('off')
plt.suptitle('Normal vs Simulated Delayed Myelination', fontsize=13)
plt.tight_layout()
cfg.paths.reports.mkdir(parents=True, exist_ok=True)
plt.savefig(cfg.paths.reports / 'mri_augment_compare.png', dpi=100)
plt.show()

In [ ]:
!git add checkpoints/ reports/ datasets/processed/
!git commit -m 'colab: 02_mri_preprocess completed'
!git push origin main